In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

from kooplearn._linalg import eigh_rank_reveal, spd_neg_pow, weighted_norm
from kooplearn.datasets import compute_prinz_potential_eig, make_prinz_potential
from kooplearn.kernel import KernelRidge


def operator_norm_error(true_operator: np.ndarray, estimated_operator: np.ndarray):
    r"""Operator norm error proxy for a Koopman estimator.

    Computes the operator norm discrepancy between the true action
    :math:`A_\pi S` and the estimated action :math:`S \widehat{G}`:

    .. math::

        \mathcal{E}(\widehat{G}) := \|A_\pi S - S \widehat{G}\|.

    Since "kooplearn" does not currently expose :math:`A_\pi` or the embedding
    operator :math:`S` explicitly, this function works with their actions on a
    common finite-dimensional representation. In practice, the caller should pass
    matrices or vectors representing the two quantities to be compared.
    """
    true_operator = np.asanyarray(true_operator)
    estimated_operator = np.asanyarray(estimated_operator)

    if true_operator.shape != estimated_operator.shape:
        raise ValueError(
            "true_operator and estimated_operator must have the same "
            f"shape, got {true_operator.shape} and "
            f"{estimated_operator.shape}."
        )

    diff = true_operator - estimated_operator
    if diff.ndim == 1:
        return float(np.linalg.norm(diff))
    return float(np.linalg.norm(diff, ord=2))


def metric_distortion(psi, C):
    r"""Empirical metric distortion :math:`\widehat\eta_i = \|\widehat\psi_i\|_{\mathcal H} /
    \sqrt{\langle \widehat C \widehat\psi_i, \widehat\psi_i\rangle}`.

    Parameters
    ----------
    psi : ndarray, shape (n,) or (n, k)
        Eigenfunction(s) evaluated at the *training* points. If 2D, each
        column is treated as a separate eigenfunction (see `weighted_norm`).
    C : ndarray, shape (n, n)
        Empirical (kernel-based) covariance, i.e. ``model.kernel_X / n_samples``.
    """
    psi = np.asarray(psi)
    n = C.shape[0]

    # ||psi||_H via the reproducing property: needs the *inverse* Gram, since
    # C = K_X / n is the Gram-based covariance, not the RKHS metric itself.
    C_inv = spd_neg_pow(C * n, exponent=-1.0)  # i.e. K_X^{-1}
    rkhs_norm = weighted_norm(psi, M=C_inv)

    # <C psi, psi> = (1/n)||psi(X)||_2^2, i.e. weighted_norm with M=None, squared, over n
    empirical_norm = weighted_norm(psi, M=None) / np.sqrt(n)

    with np.errstate(divide="ignore", invalid="ignore"):
        eta = rkhs_norm / empirical_norm
    eta = np.where(empirical_norm > 0, eta, np.nan)
    return eta if psi.ndim == 2 else float(eta)


def spectral_bias(eigenfunction, C, rho):
    r"""Empirical spectral bias :math:`\hat s_i = \widehat\eta_i \, \rho_{r+1}`."""
    eta = metric_distortion(eigenfunction, C)
    s_hat = eta * rho
    return float(s_hat), eta


def _top_sv(C, r):
    """(r+1)-st eigenvalue of a symmetric PSD matrix, via eigh_rank_reveal."""
    raw_vals, raw_vecs = np.linalg.eigh(np.asarray(C))
    _, top_vals, _ = eigh_rank_reveal(raw_vals, raw_vecs, rank=r + 1)
    if len(top_vals) <= r:
        return 0.0
    return float(top_vals[-1])


# --- truncation helpers ---
def pcr_truncation(C, r):
    r""":math:`\rho_{r+1}(\widehat G^{PCR}) = \sigma_{r+1}(\widehat C)`."""
    return _top_sv(C, r)


# kDMD uses the same (r+1)-st eigenvalue of the empirical covariance as PCR
kdmd_truncation = pcr_truncation


def rrr_truncation(C, T, r, cutoff=None):
    r""":math:`\rho_{r+1}(\widehat G^{RRR}) = \sigma_{r+1}(\widehat C^{-1/2}\widehat T)`."""
    C_inv_sqrt = spd_neg_pow(np.asarray(C), exponent=-0.5, cutoff=cutoff)
    A = C_inv_sqrt @ np.asarray(T)
    svals = np.linalg.svd(A, compute_uv=False)
    if r >= len(svals):
        return 0.0
    return float(svals[r])


# --- spectral gap (top-two magnitude eigenvalues) ---
def spectral_gap(eigenvalues):
    mags = np.sort(np.abs(eigenvalues))[::-1]
    return float(mags[0] - mags[1]) if len(mags) > 1 else np.nan


# --- spurious eigenvalues vs reference ---


def spurious_eigenvalues(est, ref, delta):
    dist = np.abs(est[:, None] - ref[None, :])
    return int(np.sum(dist.min(axis=1) > delta))


# --- compilation function for analysing spectral metrics ---


def analyse_spectrum(records, out_prefix="spectral_bias"):
    df = pd.DataFrame(records).copy()

    summary = df.groupby(["kernel", "method", "eigenfunction_id"], as_index=False).agg(
        n=("spectral_bias", "size"),
        bias_mean=("spectral_bias", "mean"),
        bias_std=("spectral_bias", "std"),
        dist_mean=("metric_distortion", "mean"),
        trunc_mean=("truncation", "mean"),
    )

    corr_df = (
        df.groupby(["kernel", "method"])
        .apply(lambda g: g["spectral_bias"].corr(g["spectral_gap"]))
        .rename("bias_gap_corr")
        .reset_index()
    )

    summary.to_csv(f"{out_prefix}_summary.csv", index=False)
    df.to_csv(f"{out_prefix}_records.csv", index=False)
    corr_df.to_csv(f"{out_prefix}_corr.csv", index=False)

    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    for (kernel, method), g in df.groupby(["kernel", "method"]):
        ax.scatter(
            g["spectral_bias"], g["spectral_gap"], s=20, alpha=0.7, label=f"{kernel} / {method}"
        )
    ax.set_xlabel("Spectral bias")
    ax.set_ylabel("Spectral gap")
    ax.legend(frameon=False, fontsize=8)
    ax.set_title("Spectral bias vs Spectral gap")
    fig.tight_layout()
    fig.savefig(f"{out_prefix}_scatter.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    return df, summary, corr_df


In [2]:
x = np.linspace(-2, 2, 2048 + 1)
gamma = 1.0
sigma = 2.0
dt = 1e-4
subsample = 100
n_train = 5000
n_trials = 10
n_show = 3

data = make_prinz_potential(X0=0, n_steps=int(7e5), gamma=gamma, sigma=sigma, random_state=0)
data = data.iloc[::subsample][:n_train]

vals_ref = compute_prinz_potential_eig(gamma, sigma, dt, num_components=5)

trial_records = []
modes_records = []
for method, reduced_rank in zip(
    ["Principal Components (kDMD)", "Reduced Rank"],
    [False, True],
):
    for trial in tqdm(range(n_trials), desc=method):
        model = KernelRidge(
            n_components=5,
            reduced_rank=reduced_rank,
            gamma=12.5,
            kernel="rbf",
            alpha=1e-6,
            random_state=trial,  # vary the seed across trials
        )
        model.fit(data)

        n = model.kernel_X_.shape[0]
        C = model.kernel_X_ / n
        T = model.kernel_YX_ / n

        vals_hat, funcs_hat = model.eig(eval_right_on=model.X_fit_[:-1])
        sort_perm = np.flip(np.argsort(np.abs(vals_hat)))
        vals_hat, funcs_hat = vals_hat[sort_perm], funcs_hat[:, sort_perm]

        gap = spectral_gap(vals_hat)
        n_spur = spurious_eigenvalues(vals_hat, vals_ref, delta=0.05)

        trial_records.append(
            {
                "kernel": "rbf",
                "method": method,
                "trial": trial,
                "spurious_eig_count": int(n_spur),
                "spectral_gap": gap,
            }
        )

        r = model.rank_  # model fitted truncation rank
        rho = (
            kdmd_truncation(C, r)
            if method == "Principal Components (kDMD)"
            else rrr_truncation(C, T, r)
        )

        modes = model.dynamical_modes(data)
        n_modes = modes.n_modes
        for j in range(n_modes):
            eigenfunction = funcs_hat[:, j]
            bias, distortion = spectral_bias(eigenfunction, C, rho)
            modes_records.append(
                {
                    "kernel": "rbf",
                    "method": method,
                    "trial": trial,
                    "eigenfunction_id": j + 1,
                    "spectral_bias": float(bias),
                    "metric_distortion": float(distortion),
                    "truncation": float(rho),
                    "spectral_gap": gap,
                    "est_eig_real": float(np.real(vals_hat[j])),
                    "est_eig_imag": float(np.imag(vals_hat[j])),
                }
            )

df, summary, corr_df = analyse_spectrum(modes_records)
print(df)
print(summary)
print(corr_df)
print(pd.DataFrame(trial_records))


Reduced Rank: 100%|██████████| 10/10 [26:27<00:00, 158.72s/it]


   kernel                       method  trial  eigenfunction_id  \
0     rbf  Principal Components (kDMD)      0                 1   
1     rbf  Principal Components (kDMD)      0                 2   
2     rbf  Principal Components (kDMD)      0                 3   
3     rbf  Principal Components (kDMD)      0                 4   
4     rbf  Principal Components (kDMD)      0                 5   
..    ...                          ...    ...               ...   
95    rbf                 Reduced Rank      9                 1   
96    rbf                 Reduced Rank      9                 2   
97    rbf                 Reduced Rank      9                 3   
98    rbf                 Reduced Rank      9                 4   
99    rbf                 Reduced Rank      9                 5   

    spectral_bias  metric_distortion  truncation  spectral_gap  est_eig_real  \
0        0.581365          11.980713    0.048525      0.063053      0.993139   
1        0.611523          12.60219

In [ ]:
import itertools

x = np.linspace(-4, 4, 1025)[:, None]
gamma = 1.0
sigma = 2.0
dt = 1e-4
subsample = 50
n_train = 500
n_trials = 1
n_show = 3


def simulate_ou(n_steps, random_state, x0=0.0):
    rng = np.random.default_rng(random_state)
    x = np.empty(n_steps, dtype=float)
    x[0] = float(x0)
    a = np.exp(-1.0)
    b = np.sqrt(1.0 - np.exp(-2.0))
    for t in range(n_steps - 1):
        x[t + 1] = a * x[t] + b * rng.standard_normal()
    return pd.DataFrame({"x": x})
def compute_ou_eig(gamma, lag, num_components):
    """Analytic Koopman eigenvalues for a discrete-time OU process at lag Dt."""
    n = np.arange(num_components)
    return np.exp(-n * gamma * lag)

data = simulate_ou(
    n_steps=int(7e5),
    random_state=0,
    x0=0.0,
)
data = data.iloc[::subsample][:n_train]

lag = dt * subsample  # the actual time step between consecutive fitted samples
vals_ref = compute_ou_eig(gamma, lag, num_components=5)

trial_records = []
modes_records = [][tuple[str, bool, str]]
for method, reduced_rank, kernel in itertools.product(
    ["PCR", "Reduced Rank"],
    [False, True],
    ["good", "bad", "ugly"],
):
    for trial in tqdm(range(n_trials), desc=method):
        model = KernelRidge(
            n_components=5,
            reduced_rank=reduced_rank,
            gamma=12.5,
            kernel=kernel,
            alpha=1e-6,
            random_state=trial,  # vary the seed across trials
        )
        model.fit(data)

        n = model.kernel_X_.shape[0]
        C = model.kernel_X_ / n
        T = model.kernel_YX_ / n

        vals_hat, funcs_hat = model.eig(eval_right_on=model.X_fit_[:-1])
        sort_perm = np.flip(np.argsort(np.abs(vals_hat)))
        vals_hat, funcs_hat = vals_hat[sort_perm], funcs_hat[:, sort_perm]

        gap = spectral_gap(vals_hat)
        n_spur = spurious_eigenvalues(vals_hat, vals_ref, delta=0.05)

        trial_records.append(
            {
                "kernel": kernel,
                "method": method,
                "trial": trial,
                "spurious_eig_count": int(n_spur),
                "spectral_gap": gap,
            }
        )

        r = model.rank_  # model fitted truncation rank
        rho = rrr_truncation(C, T, r) if method == "Reduced Rank" else pcr_truncation(C, r)

        modes = model.dynamical_modes(data)
        n_modes = modes.n_modes
        for j in range(n_modes):
            eigenfunction = funcs_hat[:, j]
            bias, distortion = spectral_bias(eigenfunction, C, rho)
            modes_records.append(
                {
                    "kernel": kernel,
                    "method": method,
                    "trial": trial,
                    "eigenfunction_id": j + 1,
                    "spectral_bias": float(bias),
                    "metric_distortion": float(distortion),
                    "truncation": float(rho),
                    "spectral_gap": gap,
                    "est_eig_real": float(np.real(vals_hat[j])),
                    "est_eig_imag": float(np.imag(vals_hat[j])),
                }
            )

df, summary, corr_df = analyse_spectrum(modes_records)
print(df)
print(summary)
print(corr_df)
print(pd.DataFrame(trial_records))


['U_', 'V_', 'X_fit_', 'feature_names_in_', 'gamma_', 'kernel_X_', 'kernel_YX_', 'kernel_Y_', 'n_features_in_', 'rank_', 'y_fit_']